Cell 1 (code): GPU and environment check, then pinned install. If pip changes numpy, restart the runtime.

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda", torch.version.cuda)

!pip install -q pycolmap==4.2.1


Cell 2 (code): confirm the install.

In [ ]:
import importlib.metadata as md
print("pycolmap", md.version("pycolmap"))

Cell 3 (code): print the real API. If cell 5 fails, paste me this output.

In [ ]:
import pycolmap
for fn in ["extract_features", "match_sequential", "incremental_mapping", "undistort_images"]:
    print("=" * 20, fn)
    print(getattr(pycolmap, fn).__doc__)
print("ImageReaderOptions:", [m for m in dir(pycolmap.ImageReaderOptions) if not m.startswith("_")])
print("CameraMode:", list(pycolmap.CameraMode.__members__))
print("SequentialPairingOptions:", [m for m in dir(pycolmap.SequentialPairingOptions) if not m.startswith("_")])


Cell 4 (code): mount Drive and import the module. Edit the three paths.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/<your folder>")        # folder containing sfm_pipeline.py
import importlib, sfm_pipeline
importlib.reload(sfm_pipeline)                                     # re-run after editing the module

FRAMES_SRC = "/content/drive/MyDrive/<your folder>/frames/<video_name>"   # Stage 0 output
DRIVE_OUT  = "/content/drive/MyDrive/<your folder>/colmap"                # results saved here
WORK       = "/content/sfm_run"                                           # fast local Colab disk

N_FRAMES     = 40        # <- the one parameter: None = all frames
CAMERA_MODEL = "OPENCV"  # iPhone 0.5x ultrawide
OVERLAP      = 20        # frames matched per frame; keep below N_FRAMES


Cell 5 (code): run the pipeline. It runs extraction, sequential matching and incremental mapping, prints the registration summary, and writes all outputs to WORK.

In [ ]:
rec, summary = sfm_pipeline.run_sfm(
    FRAMES_SRC, WORK,
    n_frames=N_FRAMES, camera_model=CAMERA_MODEL, overlap=OVERLAP,
)

Cell 6 (code): save to Drive before the session ends.

In [ ]:
import shutil
from pathlib import Path
dest = Path(DRIVE_OUT) / (f"run_{N_FRAMES}frames" if N_FRAMES else "run_full")
shutil.copytree(WORK, dest, dirs_exist_ok=True)
print("saved to", dest)
!ls "{dest}"


WORK includes database.db and the copied frames. Delete those from Drive if space is tight.

Cell 7 (code): inspect the output layout. This shows how undistorted/ is arranged, which I haven't verified.

In [ ]:
!find /content/sfm_run -maxdepth 3 -not -name "*.jpg" | sort